# 05 - FISPACT activation studies

**~20 minutes.**

### The story

The transport run is done and signed off. Now the question changes: the machine has been
operated for years, it is shut down, and somebody has to go inside to replace a component.
**What is still radioactive, how much heat does it release, and which nuclides are to blame?**

That is an activation calculation, and it is FISPACT-II's job. F4Enix sits on both ends of it:
it prepares the neutron spectrum and the irradiation history that FISPACT needs, and it reads
the output back, so the answer comes out as a table instead of a 60 000-line text file.

### What you will use

`Spectra`, `GROUP_STRUCTURES`, `Pulse`, `IrradiationScenario`, `Nuclide`, `TCF_Computer`,
`FispactOutput`, `PathwayCollection`.

> **Solutions notebook.** Identical to `notebooks/05_fispact_activation_studies.ipynb`, with the exercises filled in.
> Try them yourself first.

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

In [ ]:
import numpy as np
import pandas as pd

from f4enix.core.constants import TIME_UNITS
from f4enix.core.egroups import GROUP_STRUCTURES
from f4enix.core.irradiation import IrradiationScenario, Nuclide, Pulse, TCF_Computer
from f4enix.core.spectra import Spectra
from f4enix.output.fispact_legacy_out import FispactOutput, PathwayCollection
from f4enix.output.mctal import Mctal

FISP = DATA / "fispact"      # the FISPACT files used by this notebook
OUT_FISP = OUT / "fispact"   # everything this notebook writes
OUT_FISP.mkdir(exist_ok=True)

## 1. The activation workflow

An activation study is a chain, and FISPACT-II is only the middle of it:

```
MCNP neutron spectrum ---> fluxes file ---> collapse.i ---> collapsed cross sections
                                       \--> condense.i ---> condensed decay data
                                                                    |
                    irradiation history ---> FISPACT-II inventory run
                                                                    |
                                                            the .out file
                                                                    |
                          shutdown dose rate  |  decay heat  |  decay pathways
```

F4Enix covers the two ends: the **preparation** of the spectrum and of the irradiation
history, and the **post-processing** of the output. The inventory input itself - the material
masses, the `DOSE`, `HALF`, `HAZARDS` keywords - is not written by F4Enix. It is normally
written by hand, or with `pypact`, which F4Enix uses under the hood to read it back.

## 2. The neutron spectrum, from the mctal

FISPACT wants one flux value per energy group. Tally 4 of the ITER_1D run is exactly that:
a cell flux binned in energy. This is the same `Mctal` object as in notebook 03, and the
`Total` rows must go before anything else happens.

In [ ]:
mctal = Mctal(DATA / "iter_1d" / "mctal")
mctal.remove_totals(4)
flux = mctal.tallydata[4]

cell = 53                                   # a cell deep inside the blanket
spectrum = flux[flux["Cells"] == cell]

print("energy groups:", len(spectrum))
spectrum.head()

175 groups is not a coincidence: the tally was set up on **VITAMIN-J-175**, one of the
group structures FISPACT understands. `GROUP_STRUCTURES` carries twenty of them, always in
**eV** and always as bin *boundaries*, so a 175-group structure has 176 numbers.

In [ ]:
ebins = GROUP_STRUCTURES["VITAMIN-J-175"]

print("some structures :", sorted(GROUP_STRUCTURES)[:6], "...")
print("boundaries      :", len(ebins), "-> groups:", len(ebins) - 1)
# the mctal reports the upper bound of each group, in MeV
print("same bins as t4 :", np.allclose(spectrum["Energy"] * 1e6, ebins[1:], rtol=1e-3))

`Spectra` expects the values **normalised to 1** - the absolute intensity comes later, from
the irradiation scenario. It will normalise for you and log a warning; doing it explicitly is
cleaner and shows what is happening.

> **Careful:** energies in eV, values normalised, and one value fewer than boundaries. Getting
> any of the three wrong raises a `ValueError` or, worse, silently shifts the whole spectrum.

In [ ]:
values = spectrum["Value"].to_numpy()
mcnp_spectrum = Spectra(ebins, values / values.sum(), name=f"ITER1D_C{cell}")

print("name   :", mcnp_spectrum.name)
print("groups :", len(mcnp_spectrum.spectra_values))
print("sum    :", mcnp_spectrum.spectra_values.sum())

## 3. Comparing with a reference spectrum

A FISPACT `fluxes` file contains only the numbers - no energies. So `from_fispact` needs you to
say which group structure they belong to. Get that wrong and the file still reads perfectly,
which is why the group structure is worth double-checking.

In [ ]:
reference = Spectra.from_fispact(GROUP_STRUCTURES["VITAMIN-J-175"], FISP / "FLUXESS1")

print("name written in the file:", reference.name)
print("groups                  :", len(reference.spectra_values))

`plot` draws one or several spectra together. `lethargy=True` divides each group by
`ln(E_hi/E_lo)`, which removes the effect of the group widths - that is the shape you actually
want to compare, and the only one in which the 14 MeV peak looks like a peak.

In [ ]:
fig, ax = mcnp_spectrum.plot(lethargy=True, add_spectra=[reference])

## 4. The pre-processing files

Four files, four one-liners. `fluxes` is the spectrum for the collapse; `arb_flux` is the same
spectrum in the format the group-conversion routine wants; `collapse.i` and `condense.i` are
the two small FISPACT runs that prepare the collapsed cross sections and the decay data.

In [ ]:
mcnp_spectrum.to_fispact_fluxes(OUT_FISP)
mcnp_spectrum.to_arb_flux_file(OUT_FISP)
mcnp_spectrum.print_collapse_inp(OUT_FISP)
mcnp_spectrum.print_condensed_inp(OUT_FISP)

print(sorted(p.name for p in OUT_FISP.iterdir()))

> **Careful:** the fluxes file is named after the **spectrum**, not `FLUXES`. Name the
> spectrum after the file your FISPACT input expects, or rename it afterwards.

In [ ]:
print((OUT_FISP / "collapse.i").read_text())

`GETXS 1 175` is the line that ties the collapse to the group structure of the flux above.

For a tungsten component the resonance self-shielding matters, and `print_collapse_inp` can add
the whole `PROBTAB` / `SSFCHOOSE` / `SSFMASS` block for you - that is what Exercise 1 is about.

## 5. The irradiation history

A `Pulse` is a length of time at a given source intensity. A pulse with intensity `0` is a
dwell, and that is how you describe a pulsed machine. An `IrradiationScenario` is just the
ordered list of them.

In [ ]:
pulsed = IrradiationScenario(
    [Pulse(2, 2.68e18, TIME_UNITS.YEAR)] + [Pulse(3290, 0), Pulse(400, 5.0e19)] * 17,
    name="two_years_then_pulses",
)
pulsed.get_collapsed_table()

`get_collapsed_table` folds repeated sequences into a single row with a multiplier, which is
the only way a scenario with a few hundred pulses is readable.

Most of the time you do not type the scenario at all: it already exists in a FISPACT input, or
in a legacy `d1stime` script.

In [ ]:
scenario = IrradiationScenario.from_fispact(FISP / "inp_fispact.i")

print("pulses:", len(scenario.pulses))
scenario.get_collapsed_table()

Cooling times are given **after** shutdown, and are cumulative by default. F4Enix builds a
short label for each of them, and those labels follow the scenario everywhere.

`to_ascii` writes the whole thing to a readable `.irr` file that `from_ascii` reads back - a
much better thing to put under version control than a FISPACT input.

In [ ]:
scenario.set_cooling_times([(1, TIME_UNITS.HOUR), (1, TIME_UNITS.DAY), (1, TIME_UNITS.YEAR)])
print("cooling labels:", scenario.cooling_labels)

scenario.to_ascii(OUT_FISP / "scenario.irr")
print((OUT_FISP / "scenario.irr").read_text())

## 6. Time correction factors

In the D1S method the decay is not simulated: the photon yield of each daughter is scaled by a
**time correction factor**, the number of atoms present at a given cooling time per unit
production rate. It depends only on the half life of the daughter and on the irradiation
history - so it can be computed here, before any transport is run.

In [ ]:
tcf = TCF_Computer()
nuclides = [Nuclide.from_formula("Co60"), Nuclide.from_formula("Mn56")]

for nuclide in nuclides:
    print(f"{nuclide.write_to_formula():6s} half life: {tcf.get_half_life(nuclide):.3e} s")

In [ ]:
factors = tcf.compute_correction_factors(scenario, nuclides, norm=1.0)

pd.DataFrame(
    factors,
    index=scenario.cooling_labels,
    columns=[n.write_to_formula() for n in nuclides],
)

Mn56 has a half life of a couple of hours: one day after shutdown it is gone. Co60 has 5.3
years and barely moves over the same range. That contrast is the whole reason cooling times are
chosen the way they are.

> **Note.** These factors are what goes into a D1S-UNED `irrad` file. F4Enix writes it with
> `IrradiationFile.from_irradiation_schedules`, which is a topic of its own.

## 7. Reading a FISPACT output

A legacy FISPACT-II `.out` is a few tens of MB of fixed-format text holding every time step of
the inventory. `FispactOutput` parses the inventory through `pypact` and, on top of that, pulls
out the dose, the decay heat and the reaction pathways.

You pass the labels of the cooling times: they are attached to the **last** time steps of the
file, in order.

In [ ]:
fispact_out = FispactOutput(FISP / "testSS.out", cooling_times=["1e2", "1e4"])

print("time steps in the file :", len(fispact_out.inventory_data))
print("labelled cooling times :", fispact_out.sddr["cooling time"].unique().tolist())

`sddr` and `decay_heat` are plain DataFrames, one row per isotope per cooling time, already
sorted by contribution. `isotope % dose` is the share of that isotope, `Cumulative dose sum`
the running total - so the first rows are the ones that matter.

In [ ]:
fispact_out.sddr.head()

In [ ]:
fispact_out.decay_heat.head()

## 8. Who drives the dose

`filter_by_cum_dose` cuts the table where the cumulative contribution reaches the percentage
you ask for. This is the answer to "which nuclides do I need to worry about", and it is usually
five or six of them.

In [ ]:
top_dose = fispact_out.filter_by_cum_dose(95, "1e2")
top_dose[["element", "isotope", "dose", "isotope % dose", "Cumulative dose sum"]]

Knowing *which* nuclide is only half the story - you also want to know **which reaction made
it**, because that is what you can act on in the material specification. `add_pathways=True`
explodes every isotope row into one row per production pathway.

In [ ]:
with_paths = fispact_out.filter_by_cum_dose(95, "1e2", add_pathways=True)
with_paths[["element", "isotope", "isotope % dose", "pathway", "pathway % dose"]]

The pathways come from the `GENERIC PATHWAYS` block of the output. They are parsed
independently by `PathwayCollection`, and each `Pathway` knows its parent, its intermediates and
its reactions. `reduce()` strips the isomeric transitions, which are not real steps, and
`get_MT()` gives the ENDF reaction number of the first reaction - the number a D1S `reac` file
needs.

In [ ]:
pathways = PathwayCollection.from_file(FISP / "testSS.out")
print("pathways found:", len(pathways.pathways))

path = pathways.pathways[13]
print("as written :", path)
print("multistep  :", path.is_multistep())
print("reduced    :", path.reduce())
print("first MT   :", path.get_MT())

In [ ]:
pathways.to_dataframe().head(10)

---

# Exercises

## Exercise 1 - a collapse for a tungsten component

The shielding block in cell **22** is tungsten, so its collapse needs the resonance
self-shielding treatment.

1. take the energy-binned flux of **cell 22** from the same tally 4 DataFrame (`flux`);
2. build a `Spectra` out of it, normalised, named **`"ITER1D_C22"`**;
3. write the `fluxes` file and the collapse input into `EX1`, asking for the **tungsten**
   treatment;
4. print the `GETXS` line of the collapse input you just wrote.

**Hints**
* the group structure is the same `ebins` as above;
* `print_collapse_inp` takes a second, boolean argument;
* the `GETXS` line is the third line of `collapse.i`.

In [ ]:
EX1 = OUT / "fispact_ex1"
EX1.mkdir(exist_ok=True)

# 1: get the energy-binned flux of cell 22
cell22 = flux[flux["Cells"] == 22]

# 2: build the normalised Spectra, named "ITER1D_C22"
values_22 = cell22["Value"].to_numpy()
spectra_22 = Spectra(ebins, values_22 / values_22.sum(), name="ITER1D_C22")

# 3: write the fluxes file and the collapse input, with the tungsten treatment
spectra_22.to_fispact_fluxes(EX1)
spectra_22.print_collapse_inp(EX1, tungsten=True)

# 4: print the GETXS line of the collapse input
print((EX1 / "collapse.i").read_text().splitlines()[2])


In [ ]:
# --- self-check, do not modify ---
assert (EX1 / "ITER1D_C22").is_file(), (
    "no fluxes file called ITER1D_C22 - remember it is named after the spectra"
)
collapse_text = (EX1 / "collapse.i").read_text()
assert "SSFCHOOSE" in collapse_text and "W186" in collapse_text, (
    "the collapse input has no self-shielding block - did you ask for tungsten?"
)
assert "GETXS 1 175" in collapse_text, (
    f"unexpected GETXS line: {collapse_text.splitlines()[2]!r}"
)
assert abs(spectra_22.spectra_values.sum() - 1) < 1e-6, (
    f"the spectra sums to {spectra_22.spectra_values.sum():.4f}, it should be 1"
)
print("Exercise 1 correct")

## Exercise 2 - the decay heat drivers after a long cooling

The heat load on the hot cell handling equipment is what matters at the longer cooling time.

1. from `fispact_out`, take the isotopes making up **95% of the decay heat** at the cooling time
   labelled **`"1e4"`**, *with* their pathways, and store the table in `heat_drivers`;
2. print the pathway with the largest `pathway % heat` and the isotope it feeds.

**Hints**
* `filter_by_cum_dose` has a twin for the heat, with the same signature;
* the `pathway` columns only appear if you ask for them;
* the table comes back sorted, so `.iloc[0]` is the top row.

In [ ]:
# 1: 95% of the decay heat at "1e4", with the pathways
heat_drivers = fispact_out.filter_by_cum_heating(95, "1e4", add_pathways=True)

# 2: print the top pathway and the isotope it feeds
top = heat_drivers.iloc[0]
print(f"{top['element']}{top['isotope']} comes mostly from {top['pathway']}")
print(f"that single pathway is {top['pathway % heat']:.1f}% of the decay heat")

heat_drivers[["element", "isotope", "isotope % heat", "pathway", "pathway % heat"]]


In [ ]:
# --- self-check, do not modify ---
assert len(heat_drivers) > 0, "the filtered table is empty"
assert "pathway" in heat_drivers.columns, (
    "no pathway column - did you ask for the pathways?"
)
assert set(heat_drivers["cooling time"]) == {"1e4"}, (
    f"the table holds the cooling times {set(heat_drivers['cooling time'])}, expected 1e4"
)
assert heat_drivers["Cumulative heat sum"].max() >= 95, (
    f"only {heat_drivers['Cumulative heat sum'].max():.1f}% of the heat is covered"
)
print("Exercise 2 correct")

---

## What you can now do

| Question | Method |
|----------|--------|
| turn an MCNP spectrum into a FISPACT flux | `Spectra`, `GROUP_STRUCTURES`, `to_fispact_fluxes` |
| read a spectrum back from a `fluxes` file | `Spectra.from_fispact` |
| feed a group conversion instead | `Spectra.to_arb_flux_file` |
| compare two spectra | `Spectra.plot(lethargy=True, add_spectra=[...])` |
| write the collapse and condense inputs | `print_collapse_inp`, `print_condensed_inp` |
| self-shield a tungsten component | `print_collapse_inp(..., tungsten=True)` |
| describe an irradiation history | `Pulse`, `IrradiationScenario`, `get_collapsed_table` |
| reuse a scenario from an existing input | `IrradiationScenario.from_fispact`, `from_legacy_d1stime` |
| keep a scenario under version control | `to_ascii` / `from_ascii` |
| get the D1S time correction factors | `TCF_Computer.compute_correction_factors` |
| read a FISPACT output | `FispactOutput`, `.sddr`, `.decay_heat`, `.inventory_data` |
| find who drives the dose or the heat | `filter_by_cum_dose`, `filter_by_cum_heating` |
| get the reaction pathways | `PathwayCollection`, `Pathway.reduce`, `Pathway.get_MT` |

## Where this goes next

The pathways and the time correction factors are the two ingredients of a **D1S-UNED** shutdown
dose rate calculation: `ReactionFile.filter_by_paths` turns the first into a `reac` file and
`IrradiationFile.from_irradiation_schedules` turns the second into an `irrad` file. F4Enix does
both, and `PathwayLibrary` even ships the pathways of past F4E activation studies so you can
skip the FISPACT run altogether. Both are covered in the documentation.

<https://f4enix.readthedocs.io/en/latest/>